In [ ]:
#!/usr/bin/env python3
"""
CDI Drought Analysis Methodology - Complete Implementation
==========================================================

This script implements the Combined Drought Index (CDI) drought analysis methodology
that compares monthly CDI values to determine drought conditions and forecast 
drought outlook percentages.

Author: Generated for CDI Analysis
Date: June 2025
"""

import numpy as np
import xarray as xr
import pandas as pd
import os
import sys
from pathlib import Path
import logging
from datetime import datetime
import json
from typing import Dict, List, Tuple, Optional
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

class CDIDroughtAnalyzer:
    """
    Complete CDI Drought Analysis Implementation
    
    This class handles the entire workflow of CDI drought analysis including:
    - NetCDF file processing
    - Drought classification
    - Bitcell consistency analysis
    - Multi-year analysis
    - Drought outlook percentage calculation
    """
    
    def __init__(self, input_file: str, config: Optional[Dict] = None):
        """
        Initialize the CDI Drought Analyzer
        
        Args:
            input_file (str): Path to NetCDF file (REQUIRED)
            config (Dict, optional): Configuration dictionary. Auto-detected if not provided.
        """
        self.input_file = input_file
        self.config = config or {}
        
        # Set defaults - will be auto-detected if not provided
        self.drought_threshold = self.config.get('drought_threshold', 0.3)
        self.start_year = self.config.get('start_year', 1998)
        self.end_year = self.config.get('end_year', 2018)
        self.output_dir = Path(self.config.get('output_dir', './cdi_drought_analysis_output'))
        
        # These will be auto-detected from NetCDF when we load the data
        self.variable_name = self.config.get('variable_name', None)
        self.time_dim = self.config.get('time_dimension', None)
        self.lat_dim = self.config.get('lat_dimension', None)
        self.lon_dim = self.config.get('lon_dimension', None)
        
        # Create output directory
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # Setup logging
        self._setup_logging()
        
        # Data storage
        self.cdi_data = None
        self.drought_binary = None
        self.consistency_results = {}
        self.outlook_percentages = {}
        
        self.logger.info(f"CDI Drought Analyzer initialized")
        self.logger.info(f"Input file: {self.input_file}")
        self.logger.info(f"Output directory: {self.output_dir}")
        
    def _setup_logging(self):
        """Set up logging configuration"""
        log_file = self.output_dir / f"cdi_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
        
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler(log_file),
                logging.StreamHandler(sys.stdout)
            ]
        )
        self.logger = logging.getLogger(__name__)
        
    def load_netcdf_data(self) -> bool:
        """
        Load CDI data from NetCDF file and auto-detect everything
        
        Returns:
            bool: True if successful, False otherwise
        """
        try:
            self.logger.info(f"Loading NetCDF file: {self.input_file}")
            
            # Load the dataset
            ds = xr.open_dataset(self.input_file)
            
            # Print available variables for user information
            self.logger.info(f"Available variables in NetCDF: {list(ds.variables.keys())}")
            self.logger.info(f"Available dimensions: {list(ds.dims.keys())}")
            
            # If variable name not specified, auto-detect it
            if self.variable_name is None:
                # Look for CDI-related variables first
                for var_name in ds.variables:
                    if any(keyword in var_name.lower() for keyword in ['cdi', 'drought']):
                        if len(ds[var_name].dims) >= 2:  # Must be at least 2D
                            self.variable_name = var_name
                            break
                
                # If still not found, just take the first multi-dimensional data variable
                if self.variable_name is None:
                    for var_name in ds.variables:
                        # Skip coordinate variables and take first data variable with 3+ dimensions
                        if var_name not in ds.dims and len(ds[var_name].dims) >= 3:
                            self.variable_name = var_name
                            break
            
            if self.variable_name is None or self.variable_name not in ds.variables:
                self.logger.error(f"Could not find variable '{self.variable_name}' in NetCDF file")
                self.logger.error(f"Available variables: {list(ds.variables.keys())}")
                return False
            
            # Get the CDI data
            self.cdi_data = ds[self.variable_name]
            self.logger.info(f"Using variable: {self.variable_name}")
            self.logger.info(f"Variable shape: {self.cdi_data.shape}")
            self.logger.info(f"Variable dimensions: {self.cdi_data.dims}")
            
            # Auto-detect dimensions from the actual data variable
            dims = list(self.cdi_data.dims)
            
            # Auto-detect time dimension
            if self.time_dim is None:
                for dim in dims:
                    if any(keyword in dim.lower() for keyword in ['time', 'date', 't']):
                        self.time_dim = dim
                        break
                # If not found by name, try to detect by dimension size (time usually has most values)
                if self.time_dim is None and len(dims) >= 3:
                    # Check which dimension has the most values (likely time)
                    dim_sizes = {dim: self.cdi_data.sizes[dim] for dim in dims}
                    # Time dimension often has the most time steps
                    largest_dim = max(dim_sizes, key=dim_sizes.get)
                    self.time_dim = largest_dim
            
            # Auto-detect latitude dimension  
            if self.lat_dim is None:
                for dim in dims:
                    if dim != self.time_dim and any(keyword in dim.lower() for keyword in ['lat', 'y']):
                        self.lat_dim = dim
                        break
                # If not found by name, pick first non-time dimension
                if self.lat_dim is None:
                    remaining_dims = [d for d in dims if d != self.time_dim]
                    if remaining_dims:
                        self.lat_dim = remaining_dims[0]
            
            # Auto-detect longitude dimension
            if self.lon_dim is None:
                for dim in dims:
                    if dim != self.time_dim and dim != self.lat_dim and any(keyword in dim.lower() for keyword in ['lon', 'x']):
                        self.lon_dim = dim
                        break
                # If not found by name, pick remaining dimension
                if self.lon_dim is None:
                    remaining_dims = [d for d in dims if d != self.time_dim and d != self.lat_dim]
                    if remaining_dims:
                        self.lon_dim = remaining_dims[0]
            
            self.logger.info(f"Detected dimensions - Time: {self.time_dim}, Lat: {self.lat_dim}, Lon: {self.lon_dim}")
            
            # Get time information and filter by year range
            if self.time_dim in self.cdi_data.dims:
                time_vals = pd.to_datetime(self.cdi_data[self.time_dim].values)
                self.logger.info(f"Time range in data: {time_vals.min()} to {time_vals.max()}")
                
                # Filter by year range
                mask = (time_vals.year >= self.start_year) & (time_vals.year <= self.end_year)
                self.cdi_data = self.cdi_data.isel({self.time_dim: mask})
                
                filtered_time = pd.to_datetime(self.cdi_data[self.time_dim].values)
                self.logger.info(f"Filtered time range: {filtered_time.min()} to {filtered_time.max()}")
            
            self.logger.info(f"Final data shape: {self.cdi_data.shape}")
            
            # Basic data statistics
            valid_data = self.cdi_data.values[~np.isnan(self.cdi_data.values)]
            if len(valid_data) > 0:
                self.logger.info(f"Data range: {np.min(valid_data):.3f} to {np.max(valid_data):.3f}")
                self.logger.info(f"Valid data points: {len(valid_data)}")
                self.logger.info(f"NaN values: {np.sum(np.isnan(self.cdi_data.values))}")
            
            return True
            
        except Exception as e:
            self.logger.error(f"Error loading NetCDF file: {str(e)}")
            return False
    
    def classify_drought_conditions(self):
        """
        Step 1: Classify drought conditions based on CDI threshold
        
        Drought Classification Rules:
        - Values ≤ 0.3 indicate drought conditions (represented as 1)
        - Values > 0.3 indicate non-drought conditions (represented as 0)
        """
        self.logger.info("Step 1: Classifying drought conditions...")
        
        # Create binary drought classification
        # drought_conditions: 1 = drought, 0 = no drought
        self.drought_binary = (self.cdi_data <= self.drought_threshold).astype(int)
        
        # Handle NaN values - drop them as per instructions
        self.drought_binary = self.drought_binary.where(~np.isnan(self.cdi_data))
        
        drought_count = int(self.drought_binary.sum())
        total_count = int((~np.isnan(self.drought_binary)).sum())
        
        self.logger.info(f"Drought threshold: {self.drought_threshold}")
        self.logger.info(f"Total drought conditions: {drought_count}")
        self.logger.info(f"Total valid observations: {total_count}")
        self.logger.info(f"Drought percentage: {drought_count/total_count*100:.2f}%")
    
    def analyze_bitcell_consistency(self, month_pairs: List[Tuple[str, str]]):
        """
        Step 2: Analyze consistency between consecutive months for each bitcell
        
        Args:
            month_pairs: List of tuples containing consecutive month pairs to analyze
                        e.g., [('April', 'May'), ('May', 'June'), ...]
        """
        self.logger.info("Step 2: Analyzing bitcell consistency...")
        
        # Convert time to pandas datetime for easier manipulation
        time_index = pd.to_datetime(self.drought_binary[self.time_dim].values)
        
        for month1, month2 in month_pairs:
            self.logger.info(f"Analyzing {month1}-{month2} consistency...")
            
            # Get month numbers
            month1_num = pd.to_datetime(month1, format='%B').month if isinstance(month1, str) else month1
            month2_num = pd.to_datetime(month2, format='%B').month if isinstance(month2, str) else month2
            
            consistency_key = f"{month1}_{month2}"
            yearly_consistency = []
            
            # Analyze each year
            for year in range(self.start_year, self.end_year + 1):
                try:
                    # Get data for specific months in the year
                    month1_mask = (time_index.year == year) & (time_index.month == month1_num)
                    month2_mask = (time_index.year == year) & (time_index.month == month2_num)
                    
                    if not month1_mask.any() or not month2_mask.any():
                        self.logger.warning(f"Missing data for {month1}/{month2} {year}")
                        continue
                    
                    month1_data = self.drought_binary.isel({self.time_dim: month1_mask})
                    month2_data = self.drought_binary.isel({self.time_dim: month2_mask})
                    
                    if month1_data.size == 0 or month2_data.size == 0:
                        continue
                    
                    # Take the first occurrence if multiple exist
                    month1_data = month1_data.isel({self.time_dim: 0})
                    month2_data = month2_data.isel({self.time_dim: 0})
                    
                    # Calculate consistency for each bitcell
                    # Consistency = 1 if both months have same drought status, 0 otherwise
                    consistency = (month1_data == month2_data).astype(int)
                    
                    # Handle NaN values - only use valid data points
                    valid_mask = ~(np.isnan(month1_data) | np.isnan(month2_data))
                    consistency = consistency.where(valid_mask)
                    
                    yearly_consistency.append({
                        'year': year,
                        'consistency': consistency,
                        'month1_data': month1_data,
                        'month2_data': month2_data
                    })
                    
                except Exception as e:
                    self.logger.warning(f"Error processing {month1}-{month2} {year}: {str(e)}")
                    continue
            
            self.consistency_results[consistency_key] = yearly_consistency
            self.logger.info(f"Processed {len(yearly_consistency)} years for {month1}-{month2}")
    
    def calculate_outlook_percentages(self, month_pairs: List[Tuple[str, str]]):
        """
        Step 3: Calculate drought outlook percentages for each bitcell
        
        Args:
            month_pairs: List of month pairs analyzed
        """
        self.logger.info("Step 3: Calculating drought outlook percentages...")
        
        for month1, month2 in month_pairs:
            consistency_key = f"{month1}_{month2}"
            
            if consistency_key not in self.consistency_results:
                self.logger.warning(f"No consistency results for {consistency_key}")
                continue
            
            yearly_data = self.consistency_results[consistency_key]
            
            if not yearly_data:
                self.logger.warning(f"No yearly data for {consistency_key}")
                continue
            
            # Stack all consistency arrays
            consistency_arrays = []
            for year_data in yearly_data:
                if 'consistency' in year_data:
                    consistency_arrays.append(year_data['consistency'])
            
            if not consistency_arrays:
                continue
            
            # Combine all years - create a 3D array (year, lat, lon)
            try:
                # Stack along a new dimension (year)
                stacked_consistency = xr.concat(consistency_arrays, dim='year')
                
                # Calculate percentage for each bitcell
                # Count valid (non-NaN) consistency values and sum them
                valid_count = (~np.isnan(stacked_consistency)).sum(dim='year')
                consistent_count = stacked_consistency.sum(dim='year', skipna=True)
                
                # Calculate percentage (avoid division by zero)
                outlook_percentage = xr.where(
                    valid_count > 0,
                    (consistent_count / valid_count) * 100,
                    np.nan
                )
                
                self.outlook_percentages[consistency_key] = {
                    'percentage': outlook_percentage,
                    'total_years': valid_count,
                    'consistent_years': consistent_count
                }
                
                # Log statistics
                valid_percentages = outlook_percentage.values[~np.isnan(outlook_percentage.values)]
                if len(valid_percentages) > 0:
                    self.logger.info(f"{consistency_key} outlook statistics:")
                    self.logger.info(f"  Mean percentage: {np.mean(valid_percentages):.2f}%")
                    self.logger.info(f"  Min percentage: {np.min(valid_percentages):.2f}%")
                    self.logger.info(f"  Max percentage: {np.max(valid_percentages):.2f}%")
                    self.logger.info(f"  Valid bitcells: {len(valid_percentages)}")
                
            except Exception as e:
                self.logger.error(f"Error calculating outlook for {consistency_key}: {str(e)}")
    
    def save_results(self):
        """
        Step 4: Save all results to files
        """
        self.logger.info("Step 4: Saving results...")
        
        # Save outlook percentages as NetCDF files
        for month_pair, data in self.outlook_percentages.items():
            try:
                # Create dataset
                ds = xr.Dataset({
                    'outlook_percentage': (['lat', 'lon'], data['percentage'].values),
                    'total_years': (['lat', 'lon'], data['total_years'].values),
                    'consistent_years': (['lat', 'lon'], data['consistent_years'].values)
                }, coords={
                    'lat': data['percentage'][self.lat_dim],
                    'lon': data['percentage'][self.lon_dim]
                })
                
                # Add attributes
                ds.attrs['title'] = f'CDI Drought Outlook Analysis - {month_pair}'
                ds.attrs['drought_threshold'] = self.drought_threshold
                ds.attrs['analysis_period'] = f'{self.start_year}-{self.end_year}'
                ds.attrs['created'] = datetime.now().isoformat()
                
                ds['outlook_percentage'].attrs['units'] = 'percent'
                ds['outlook_percentage'].attrs['long_name'] = 'Drought Outlook Consistency Percentage'
                ds['total_years'].attrs['long_name'] = 'Total Years with Valid Data'
                ds['consistent_years'].attrs['long_name'] = 'Years with Consistent Drought Status'
                
                # Save NetCDF
                output_file = self.output_dir / f'drought_outlook_{month_pair}.nc'
                ds.to_netcdf(output_file)
                self.logger.info(f"Saved NetCDF: {output_file}")
                
                # Save as CSV for easy viewing
                df = ds.to_dataframe().reset_index()
                csv_file = self.output_dir / f'drought_outlook_{month_pair}.csv'
                df.to_csv(csv_file, index=False)
                self.logger.info(f"Saved CSV: {csv_file}")
                
            except Exception as e:
                self.logger.error(f"Error saving results for {month_pair}: {str(e)}")
    
    def generate_summary_report(self):
        """
        Generate a comprehensive summary report
        """
        self.logger.info("Generating summary report...")
        
        report = {
            'analysis_config': self.config,
            'dataset_info': {
                'input_file': str(self.input_file),
                'variable_name': self.variable_name,
                'drought_threshold': self.drought_threshold,
                'analysis_period': f'{self.start_year}-{self.end_year}',
                'dataset_shape': list(self.cdi_data.shape) if self.cdi_data is not None else None
            },
            'results_summary': {}
        }
        
        # Add results summary
        for month_pair, data in self.outlook_percentages.items():
            valid_percentages = data['percentage'].values[~np.isnan(data['percentage'].values)]
            if len(valid_percentages) > 0:
                report['results_summary'][month_pair] = {
                    'mean_percentage': float(np.mean(valid_percentages)),
                    'min_percentage': float(np.min(valid_percentages)),
                    'max_percentage': float(np.max(valid_percentages)),
                    'std_percentage': float(np.std(valid_percentages)),
                    'valid_bitcells': int(len(valid_percentages))
                }
        
        # Save report
        report_file = self.output_dir / 'analysis_summary.json'
        with open(report_file, 'w') as f:
            json.dump(report, f, indent=2)
        
        self.logger.info(f"Summary report saved: {report_file}")
        
        # Create human-readable report
        text_report = self.output_dir / 'analysis_report.txt'
        with open(text_report, 'w') as f:
            f.write("CDI DROUGHT ANALYSIS REPORT\n")
            f.write("=" * 50 + "\n\n")
            f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Input File: {self.input_file}\n")
            f.write(f"Drought Threshold: {self.drought_threshold}\n")
            f.write(f"Analysis Period: {self.start_year}-{self.end_year}\n\n")
            
            f.write("RESULTS SUMMARY\n")
            f.write("-" * 20 + "\n")
            for month_pair, stats in report['results_summary'].items():
                f.write(f"\n{month_pair.replace('_', ' to ')} Analysis:\n")
                f.write(f"  Average Consistency: {stats['mean_percentage']:.2f}%\n")
                f.write(f"  Range: {stats['min_percentage']:.2f}% - {stats['max_percentage']:.2f}%\n")
                f.write(f"  Standard Deviation: {stats['std_percentage']:.2f}%\n")
                f.write(f"  Valid Bitcells: {stats['valid_bitcells']}\n")
        
        self.logger.info(f"Text report saved: {text_report}")
    
    def run_complete_analysis(self, month_pairs: List[Tuple[str, str]]):
        """
        Run the complete CDI drought analysis workflow
        
        Args:
            month_pairs: List of consecutive month pairs to analyze
        """
        self.logger.info("="*60)
        self.logger.info("STARTING CDI DROUGHT ANALYSIS")
        self.logger.info("="*60)
        
        try:
            # Step 1: Load data
            if not self.load_netcdf_data():
                self.logger.error("Failed to load NetCDF data. Aborting analysis.")
                return False
            
            # Step 2: Classify drought conditions
            self.classify_drought_conditions()
            
            # Step 3: Analyze bitcell consistency
            self.analyze_bitcell_consistency(month_pairs)
            
            # Step 4: Calculate outlook percentages
            self.calculate_outlook_percentages(month_pairs)
            
            # Step 5: Save results
            self.save_results()
            
            # Step 6: Generate summary report
            self.generate_summary_report()
            
            self.logger.info("="*60)
            self.logger.info("CDI DROUGHT ANALYSIS COMPLETED SUCCESSFULLY")
            self.logger.info("="*60)
            
            return True
            
        except Exception as e:
            self.logger.error(f"Analysis failed with error: {str(e)}")
            return False


def create_default_config():
    """
    Create a default configuration dictionary
    
    Returns:
        Dict: Default configuration parameters
    """
    return {
        # Input/Output Configuration
        'input_file': 'path/to/your/cdi_data.nc',  # Path to NetCDF file
        'output_dir': './cdi_drought_analysis_output',  # Output directory
        
        # Data Configuration
        'variable_name': 'CDI',  # Name of CDI variable in NetCDF
        'time_dimension': 'time',  # Name of time dimension
        'lat_dimension': 'lat',  # Name of latitude dimension  
        'lon_dimension': 'lon',  # Name of longitude dimension
        
        # Analysis Parameters
        'drought_threshold': 0.3,  # CDI threshold for drought classification
        'start_year': 1998,  # Start year for analysis
        'end_year': 2018,  # End year for analysis
        
        # Processing Options
        'drop_na': True,  # Drop NaN values before processing
        'save_intermediate': False,  # Save intermediate results
    }


def main():
    """
    Main execution function - SIMPLIFIED USAGE
    Only the NetCDF file path is required!
    """
    
    # ONLY REQUIRED: Path to your NetCDF file
    netcdf_file = '/Volumes/data/nacp/results/netcdf/cdi_1.nc'
    
    # OPTIONAL: Additional configuration (everything is auto-detected if not specified)
    optional_config = {
        'output_dir': './cdi_drought_analysis_results',  # Optional: custom output directory
        'drought_threshold': 0.3,                        # Optional: custom drought threshold
        # 'start_year': 2000,                           # Optional: custom start year  
        # 'end_year': 2020,                             # Optional: custom end year
        # 'variable_name': 'your_variable_name',        # Optional: if auto-detection fails
    }
    
    # Define month pairs for consistency analysis
    month_pairs = [
        ('April', 'May'),      # Primary example from documentation
        ('May', 'June'),       # You can add more pairs as needed
        ('June', 'July'),
        ('July', 'August'),
    ]
    
    print("CDI Drought Analysis - AUTO-DETECTION MODE")
    print("=" * 50)
    print(f"NetCDF file: {netcdf_file}")
    print("Everything else will be auto-detected!")
    print("\nMonth pairs to analyze:")
    for pair in month_pairs:
        print(f"  {pair[0]} -> {pair[1]}")
    print("\n" + "=" * 50)
    
    # Create analyzer instance - only needs the file path!
    analyzer = CDIDroughtAnalyzer(netcdf_file, optional_config)
    
    # Run complete analysis
    success = analyzer.run_complete_analysis(month_pairs)
    
    if success:
        print(f"\n✅ Analysis completed successfully!")
        print(f"📁 Results saved to: {analyzer.output_dir}")
        print(f"📊 Check the following files:")
        print(f"   - analysis_summary.json (detailed statistics)")
        print(f"   - analysis_report.txt (human-readable report)")
        print(f"   - drought_outlook_*.nc (NetCDF result files)")
        print(f"   - drought_outlook_*.csv (CSV result files)")
        print(f"   - *.log (analysis log file)")
    else:
        print("\n❌ Analysis failed. Check the log file for details.")
    
    return success


def simple_analysis(netcdf_file: str, output_dir: str = None):
    """
    Super simple function - just provide the NetCDF file path!
    
    Args:
        netcdf_file (str): Path to your NetCDF file
        output_dir (str, optional): Where to save results
    
    Returns:
        bool: True if successful
    """
    config = {}
    if output_dir:
        config['output_dir'] = output_dir
    
    # Default month pairs
    month_pairs = [('April', 'May'), ('May', 'June'), ('June', 'July')]
    
    analyzer = CDIDroughtAnalyzer(netcdf_file, config)
    return analyzer.run_complete_analysis(month_pairs)


if __name__ == "__main__":
    # SUPER SIMPLE USAGE - Just provide your NetCDF file path!
    
    # Method 1: Use main() function - everything auto-detected
    main()
    
    # Method 2: Even simpler - one line function call
    """
    success = simple_analysis('/Volumes/data/nacp/results/netcdf/cdi_1.nc')
    """
    
    # Method 3: Custom configuration (only if auto-detection doesn't work)
    """
    custom_config = {
        'output_dir': './my_custom_output',
        'drought_threshold': 0.25,  # Different threshold
        'start_year': 2000,
        'end_year': 2020,
        # Only specify these if auto-detection fails:
        # 'variable_name': 'your_specific_variable_name',
        # 'time_dimension': 'your_time_dim_name',
        # 'lat_dimension': 'your_lat_dim_name',
        # 'lon_dimension': 'your_lon_dim_name',
    }
    
    month_pairs = [('April', 'May'), ('October', 'November')]
    
    analyzer = CDIDroughtAnalyzer('/your/netcdf/file.nc', custom_config)
    analyzer.run_complete_analysis(month_pairs)
    """

CDI Drought Analysis - AUTO-DETECTION MODE
NetCDF file: /Volumes/data/nacp/results/netcdf/cdi_1.nc
Everything else will be auto-detected!

Month pairs to analyze:
  April -> May
  May -> June
  June -> July
  July -> August

2025-06-10 15:09:54,517 - INFO - CDI Drought Analyzer initialized
2025-06-10 15:09:54,517 - INFO - Input file: /Volumes/data/nacp/results/netcdf/cdi_1.nc
2025-06-10 15:09:54,518 - INFO - Output directory: cdi_drought_analysis_results
2025-06-10 15:09:54,518 - INFO - ============================================================
2025-06-10 15:09:54,519 - INFO - STARTING CDI DROUGHT ANALYSIS
2025-06-10 15:09:54,519 - INFO - ============================================================
2025-06-10 15:09:54,519 - INFO - Loading NetCDF file: /Volumes/data/nacp/results/netcdf/cdi_1.nc
2025-06-10 15:09:54,533 - INFO - Available variables in NetCDF: ['latitude', 'longitude', 'time', 'cdi']
2025-06-10 15:09:54,534 - INFO - Available dimensions: ['latitude', 'longitude', 'time']